In [8]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import time
import os
import warnings
warnings.filterwarnings('ignore')

# Load model (same as Hour 2)
print("Loading model weights...")
model = models.googlenet(pretrained=True)
model.eval()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Model ready on {device} ✓")

Loading model weights...
Model ready on cpu ✓


In [9]:
import urllib.request

print("Downloading ImageNet class labels...")
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt",
    "imagenet_classes.txt"
)

with open("imagenet_classes.txt") as f:
    labels = [line.strip() for line in f.readlines()]

print(f"Loaded {len(labels)} class labels ✓")
print(f"Example labels: {labels[0]}, {labels[1]}, {labels[2]}...")

Loaded 1000 class labels ✓
Example labels: tench, goldfish, great white shark...


In [14]:
# These are the keywords we look for in GoogLeNet's prediction
# If the prediction contains any of these words → PCOD Friendly
PCOD_FRIENDLY_KEYWORDS = [
    # eggs
    'egg', 'hen', 'cock', 'chicken', 'omelette',
    # fish
    'salmon', 'fish', 'trout', 'tench', 'coho',
    # salad / vegetables
    'salad', 'broccoli', 'spinach', 'cucumber', 'lettuce',
    'vegetable', 'cauliflower', 'zucchini', 'cabbage', 'kale',
    'tomato', 'pepper', 'mushroom', 'artichoke', 'asparagus',
    # fruits / berries
    'berry', 'strawberry', 'blueberry', 'raspberry', 'fruit',
    'lemon', 'orange', 'apple', 'pomegranate',
    # legumes
    'lentil', 'bean', 'pea', 'edamame', 'chickpea', 'soup',
    # nuts / healthy fats
    'almond', 'walnut', 'nut', 'avocado', 'olive',
    # other healthy
    'shrimp', 'crab', 'lobster', 'sashimi', 'miso'
]

# If the prediction contains any of these words → PCOD Unfriendly
PCOD_UNFRIENDLY_KEYWORDS = [
    'donut', 'doughnut', 'pizza', 'french fries', 'ice cream',
    'cake', 'waffle', 'fried', 'bread', 'trifle', 'chocolate',
    'candy', 'pretzel', 'bagel', 'bun', 'pastry', 'cream', 'sundae'
]

def get_pcod_label(imagenet_prediction):
    prediction_lower = imagenet_prediction.lower()
    
    for keyword in PCOD_FRIENDLY_KEYWORDS:
        if keyword in prediction_lower:
            return "PCOD-Friendly", "✅"
    
    for keyword in PCOD_UNFRIENDLY_KEYWORDS:
        if keyword in prediction_lower:
            return "PCOD-Unfriendly", "❌"
    
    # If no keyword matches, default to unfriendly to be safe
    return "PCOD-Unfriendly (uncertain)", "⚠️"

print("PCOD decision logic ready ✓")

PCOD decision logic ready ✓


In [15]:
def preprocess_image(image_path):
    transform = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    img = Image.open(image_path).convert('RGB')
    tensor = transform(img).unsqueeze(0)
    return tensor.to(device)

print("Preprocessing pipeline ready ✓")
print("  Resize     : 256px")
print("  Center crop: 224 x 224px")
print("  Normalize  : ImageNet mean/std")

Preprocessing pipeline ready ✓
  Resize     : 256px
  Center crop: 224 x 224px
  Normalize  : ImageNet mean/std


In [17]:
import os

os.chdir(r'C:\Users\samru\OneDrive\Desktop\HerCycleAI')

print("Current folder:", os.getcwd())
print()
print("Files in test_images:")
for f in os.listdir('test_images'):
    print(f"  - {f}")

Current folder: C:\Users\samru\OneDrive\Desktop\HerCycleAI

Files in test_images:
  - berries.jpg
  - cake.jpg
  - donut.jpg
  - eggs.jpg
  - fries.jpg
  - greek_salad.jpg
  - ice_cream.jpg
  - lentils.jpg
  - pizza.jpg
  - salmon.jpg


In [18]:
# Ground truth labels - what each image ACTUALLY is
# 1 = Friendly, 0 = Unfriendly (you labeled these yourself)
ground_truth = {
    'eggs.jpg'       : 1,
    'greek_salad.jpg': 1,
    'salmon.jpg'     : 1,
    'berries.jpg'    : 1,
    'lentils.jpg'    : 1,
    'donut.jpg'      : 0,
    'pizza.jpg'      : 0,
    'fries.jpg'      : 0,
    'cake.jpg'       : 0,
    'ice_cream.jpg'  : 0
}

results = []
inference_times = []

print("=" * 65)
print("HerCycle AI — Food Classifier Inference Results")
print("=" * 65)

for image_file, true_label in ground_truth.items():
    image_path = os.path.join('test_images', image_file)
    
    # Check if image exists
    if not os.path.exists(image_path):
        print(f"  ⚠️  {image_file} not found — skipping")
        continue
    
    # Preprocess
    tensor = preprocess_image(image_path)
    
    # Run inference and measure speed
    start = time.time()
    with torch.no_grad():
        output = torch.nn.functional.softmax(model(tensor), dim=1)
    elapsed_ms = (time.time() - start) * 1000
    
    # Get top prediction
    confidence, idx = torch.max(output, 1)
    imagenet_label = labels[idx.item()]
    confidence_pct = confidence.item() * 100
    
    # Apply PCOD decision logic
    pcod_label, icon = get_pcod_label(imagenet_label)
    predicted_binary = 1 if "Friendly" in pcod_label and "uncertain" not in pcod_label else 0
    correct = "✓" if predicted_binary == true_label else "✗"
    
    inference_times.append(elapsed_ms)
    results.append({
        'image': image_file,
        'imagenet_pred': imagenet_label,
        'confidence': confidence_pct,
        'pcod_label': pcod_label,
        'correct': predicted_binary == true_label,
        'time_ms': elapsed_ms
    })
    
    print(f"\n  Image      : {image_file}")
    print(f"  Detected   : {imagenet_label}")
    print(f"  Confidence : {confidence_pct:.1f}%")
    print(f"  PCOD Label : {icon} {pcod_label}")
    print(f"  Speed      : {elapsed_ms:.1f}ms")
    print(f"  Correct    : {correct}")
    print(f"  {'-' * 50}")

print(f"\n  Avg inference speed : {sum(inference_times)/len(inference_times):.1f}ms per image")
print(f"  Total images tested : {len(results)}")
print("=" * 65)

HerCycle AI — Food Classifier Inference Results

  Image      : eggs.jpg
  Detected   : corn
  Confidence : 49.1%
  PCOD Label : ⚠️ PCOD-Unfriendly (uncertain)
  Speed      : 256.0ms
  Correct    : ✗
  --------------------------------------------------

  Image      : greek_salad.jpg
  Detected   : zucchini
  Confidence : 17.2%
  PCOD Label : ✅ PCOD-Friendly
  Speed      : 226.0ms
  Correct    : ✓
  --------------------------------------------------

  Image      : salmon.jpg
  Detected   : pizza
  Confidence : 31.1%
  PCOD Label : ❌ PCOD-Unfriendly
  Speed      : 204.0ms
  Correct    : ✗
  --------------------------------------------------

  Image      : berries.jpg
  Detected   : hair slide
  Confidence : 19.9%
  PCOD Label : ⚠️ PCOD-Unfriendly (uncertain)
  Speed      : 201.0ms
  Correct    : ✗
  --------------------------------------------------

  Image      : lentils.jpg
  Detected   : hot pot
  Confidence : 35.3%
  PCOD Label : ⚠️ PCOD-Unfriendly (uncertain)
  Speed      : 212.

In [19]:
correct_count = sum(1 for r in results if r['correct'])
total = len(results)

print("=" * 65)
print("INFERENCE SUMMARY")
print("=" * 65)
print(f"  {'Image':<20} {'PCOD Label':<25} {'Conf%':<8} {'ms':<8} {'OK?'}")
print(f"  {'-'*60}")

for r in results:
    icon = "✓" if r['correct'] else "✗"
    print(f"  {r['image']:<20} {r['pcod_label']:<25} {r['confidence']:<8.1f} {r['time_ms']:<8.1f} {icon}")

print(f"\n  Correct predictions  : {correct_count}/{total}")
print(f"  Baseline accuracy    : {correct_count/total*100:.1f}%")
print(f"  Avg speed            : {sum(r['time_ms'] for r in results)/len(results):.1f}ms/image")
print(f"\n  Note: Baseline using pretrained weights only.")
print(f"  Fine-tuned model expected to exceed 80% accuracy.")
print("=" * 65)

INFERENCE SUMMARY
  Image                PCOD Label                Conf%    ms       OK?
  ------------------------------------------------------------
  eggs.jpg             PCOD-Unfriendly (uncertain) 49.1     256.0    ✗
  greek_salad.jpg      PCOD-Friendly             17.2     226.0    ✓
  salmon.jpg           PCOD-Unfriendly           31.1     204.0    ✗
  berries.jpg          PCOD-Unfriendly (uncertain) 19.9     201.0    ✗
  lentils.jpg          PCOD-Unfriendly (uncertain) 35.3     212.0    ✗
  donut.jpg            PCOD-Unfriendly           83.0     207.0    ✓
  pizza.jpg            PCOD-Unfriendly           93.5     208.0    ✓
  fries.jpg            PCOD-Unfriendly (uncertain) 16.4     202.0    ✓
  cake.jpg             PCOD-Unfriendly           48.6     196.0    ✓
  ice_cream.jpg        PCOD-Unfriendly (uncertain) 42.9     213.0    ✓

  Correct predictions  : 6/10
  Baseline accuracy    : 60.0%
  Avg speed            : 212.5ms/image

  Note: Baseline using pretrained weights only

In [20]:
# Quick diagnostic - see RAW ImageNet prediction for problematic images
problem_images = ['eggs.jpg', 'salmon.jpg', 'berries.jpg', 'lentils.jpg']

print("RAW ImageNet predictions for problem images:")
print("-" * 50)

for img_file in problem_images:
    image_path = os.path.join('test_images', img_file)
    tensor = preprocess_image(image_path)
    
    with torch.no_grad():
        output = torch.nn.functional.softmax(model(tensor), dim=1)
    
    # Show TOP 5 predictions for each image
    top5_conf, top5_idx = torch.topk(output, 5)
    
    print(f"\n{img_file}:")
    for i in range(5):
        print(f"  {i+1}. {labels[top5_idx[0][i].item()]:<30} {top5_conf[0][i].item()*100:.1f}%")

RAW ImageNet predictions for problem images:
--------------------------------------------------

eggs.jpg:
  1. corn                           49.1%
  2. cauliflower                    30.8%
  3. mashed potato                  9.2%
  4. ear                            6.9%
  5. potpie                         0.6%

salmon.jpg:
  1. pizza                          31.1%
  2. plate                          15.1%
  3. meat loaf                      4.8%
  4. zucchini                       2.6%
  5. mashed potato                  2.2%

berries.jpg:
  1. hair slide                     19.9%
  2. custard apple                  6.9%
  3. buckeye                        4.1%
  4. Granny Smith                   2.8%
  5. buckle                         2.7%

lentils.jpg:
  1. hot pot                        35.3%
  2. wok                            9.1%
  3. guacamole                      6.2%
  4. wooden spoon                   5.2%
  5. soup bowl                      5.1%
